In [1]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from keras.models import Sequential
from keras.layers import Dense, SimpleRNN, LSTM, GRU

In [2]:
data = pd.read_csv('https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv')

values = data['Passengers'].values.reshape(-1,1)

In [3]:
scaler = MinMaxScaler()
values = scaler.fit_transform(values)

In [4]:
def create_dataset(dataset, time_step=3):
    X, Y = [], []
    for i in range(len(dataset)-time_step-1):
        X.append(dataset[i:(i+time_step), 0])
        Y.append(dataset[i + time_step, 0])
    return np.array(X), np.array(Y)

time_step = 3
X, y = create_dataset(values, time_step)

In [5]:
X = X.reshape(X.shape[0], X.shape[1], 1)

In [6]:
train_size = int(len(X) * 0.7)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [11]:
def run_model(model_type):
    model = Sequential()

    if model_type == "RNN":
        model.add(SimpleRNN(50, input_shape=(time_step,1)))
    elif model_type == "LSTM":
        model.add(LSTM(50, input_shape=(time_step,1)))
    elif model_type == "GRU":
        model.add(GRU(50, input_shape=(time_step,1)))

    model.add(Dense(1))

    model.compile(optimizer='adam', loss='mse')
    # Training
    start_time = time.time()
    history = model.fit(X_train, y_train, epochs=20, batch_size=16, verbose=0)
    end_time = time.time()

    # Prediction
    preds = model.predict(X_test)

    # Inverse scaling
    preds = scaler.inverse_transform(preds)
    actual = scaler.inverse_transform(y_test.reshape(-1,1))

    # Metrics
    loss = history.history['loss'][-1]
    rmse = np.sqrt(mean_squared_error(actual, preds))
    training_time = end_time - start_time

    return loss, rmse, training_time


In [12]:
results = {}

for model_name in ["RNN", "LSTM", "GRU"]:
    print(f"Running {model_name}...")
    results[model_name] = run_model(model_name)

Running RNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
Running LSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
Running GRU...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 551ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 303ms/step


In [13]:
print("\n🔹 Results Comparison:")
for model, (loss, rmse, t) in results.items():
    print(f"{model} -> Loss: {loss:.4f}, RMSE: {rmse:.4f}, Time: {t:.2f} sec")


🔹 Results Comparison:
RNN -> Loss: 0.0037, RMSE: 72.0407, Time: 4.05 sec
LSTM -> Loss: 0.0052, RMSE: 84.6165, Time: 4.27 sec
GRU -> Loss: 0.0039, RMSE: 74.3128, Time: 4.54 sec
